In [ ]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

s = combined[["date", "Lpoly_expected_ml"]].copy()
s["date"] = pd.to_datetime(s["date"].astype(str), format="%Y%m%d")
s = s.sort_values("date").set_index("date")

# force daily frequency
s = s.asfreq("D")

ema = s["Lpoly_expected_ml"].ewm(span=30, adjust=False).mean()

s["Lpoly_filled"] = s["Lpoly_expected_ml"]
s.loc[s["Lpoly_filled"].isna(), "Lpoly_filled"] = ema

s["was_imputed"] = s["Lpoly_expected_ml"].isna()

y = s["Lpoly_filled"].values
y_norm = (y - np.mean(y)) / np.std(y)

df = pd.DataFrame({
    "t": np.arange(1, len(y_norm) + 1),        # numeric EDM time
    "x": y_norm
})

In [ ]:
vars = ["Lpoly", "temp", "salinity", "chl"]

df_mv = df[["t"] + vars].copy()